In [1]:
import json
import os
from pathlib import Path
import runpy
import shutil

# Support kernels started in kubernetes-experiments or the workspace root.
WORKSPACE_DIR = next(
    directory for directory in (Path.cwd(), *Path.cwd().parents)
    if (directory / "kubernetes-launcher/launch.sh").is_file()
)
EXPERIMENTS_DIR = WORKSPACE_DIR / "kubernetes-experiments"
validate_config = runpy.run_path(
    str(WORKSPACE_DIR / "kubernetes-launcher/render-experiment.py")
)["validate_config"]


In [2]:
CLEAN_EXISTING_FILES = True

list_replicas = [4, 8, 12]
list_clients = [4, 8, 12]

# Generate a full configuration sweep for each total pacing value.
list_throughput = [128]

# Frequency is a pacing parameter; actual duration/throughput also depend on latency.
EXPECTED_DURATION = 60
FAILURE_TIME = EXPECTED_DURATION // 4  # Seconds after the client Job starts.

CLIENT_IMAGE = "moirai-client:dev"

SIMULATION_DIR = EXPERIMENTS_DIR / "simulation-files"
if CLEAN_EXISTING_FILES:
    if SIMULATION_DIR.is_symlink():
        SIMULATION_DIR.unlink()
    elif SIMULATION_DIR.exists():
        shutil.rmtree(SIMULATION_DIR)

# Cleanup only removes generated configurations, preserving previous results.
for case in ("good-case", "faulty-case"):
    for mode in ("optimistic", "consistent", "both"):
        (SIMULATION_DIR / case / mode).mkdir(parents=True, exist_ok=True)
        (EXPERIMENTS_DIR / "results" / case / mode).mkdir(parents=True, exist_ok=True)

if len(list_replicas) != len(list_clients):
    raise ValueError("list_replicas and list_clients must have the same length")


In [3]:
def write_file(file_path, output_path, replica_image, with_fault, simulation_name, n_replicas, n_clients, frequency, n_operations, optimistic, consistent):
    config_dir = EXPERIMENTS_DIR / file_path
    results_dir = EXPERIMENTS_DIR / output_path
    config_dir.mkdir(parents=True, exist_ok=True)
    results_dir.mkdir(parents=True, exist_ok=True)
    config_path = config_dir / f"{simulation_name}.json"

    config_dict = {
        "simulation_name": simulation_name,
        "n_replicas": n_replicas,
        "n_clients": n_clients,
        "n_operations": n_operations,
        "frequency": frequency,
        "with_fault": with_fault,
        "failure_time": FAILURE_TIME,
        "use_optimistic_requests": optimistic,
        "use_consistent_requests": consistent,
        "replica_image": replica_image,
        "client_image": CLIENT_IMAGE,
        # launch.sh resolves this relative to the JSON file, not the notebook.
        "output_path": os.path.relpath(results_dir, start=config_dir),
    }
    config_path.write_text(json.dumps(config_dict, indent=2) + "\n", encoding="utf-8")
    # Use the launcher's validation so schema and value constraints stay in sync.
    validate_config(config_path)


### AW-Set Node

In [4]:
for throughput in list_throughput:
    for i in range(len(list_replicas)):
        n_replicas = list_replicas[i]
        n_clients = list_clients[i]

        frequency = throughput // n_clients
        n_operations = frequency*EXPECTED_DURATION

        ######################
        ## Commit Good-Case ##
        ######################

        replica_image = "moirai-replica-commit:dev"
        with_fault = False
        simulation_name = f"commit-awset-{n_replicas}r-{n_clients}c-{throughput}f"

        output_path = "results/good-case/optimistic/"
        file_path = "simulation-files/good-case/optimistic/"
        write_file(file_path, output_path, replica_image, with_fault, simulation_name, n_replicas, n_clients, frequency, n_operations, True, False)

        output_path = "results/good-case/consistent/"
        file_path = "simulation-files/good-case/consistent/"
        write_file(file_path, output_path, replica_image, with_fault, simulation_name, n_replicas, n_clients, frequency, n_operations, False, True)

        output_path = "results/good-case/both/"
        file_path = "simulation-files/good-case/both/"
        write_file(file_path, output_path, replica_image, with_fault, simulation_name, n_replicas, n_clients, frequency//2, n_operations//2, True, True)

        #######################
        ## Vanilla Good-Case ##
        #######################

        replica_image = "moirai-replica-vanilla:dev"
        with_fault = False
        simulation_name = f"vanilla-awset-{n_replicas}r-{n_clients}c-{throughput}f"

        output_path = "results/good-case/optimistic/"
        file_path = "simulation-files/good-case/optimistic/"
        write_file(file_path, output_path, replica_image, with_fault, simulation_name, n_replicas, n_clients, frequency, n_operations, True, False)

        output_path = "results/good-case/consistent/"
        file_path = "simulation-files/good-case/consistent/"
        write_file(file_path, output_path, replica_image, with_fault, simulation_name, n_replicas, n_clients, frequency, n_operations, False, True)

        output_path = "results/good-case/both/"
        file_path = "simulation-files/good-case/both/"
        write_file(file_path, output_path, replica_image, with_fault, simulation_name, n_replicas, n_clients, frequency//2, n_operations//2, True, True)

        ########################
        ## Commit Faulty-Case ##
        ########################

        replica_image = "moirai-replica-commit:dev"
        with_fault = True
        simulation_name = f"commit-awset-{n_replicas}r-{n_clients}c-{throughput}f"

        output_path = "results/faulty-case/optimistic/"
        file_path = "simulation-files/faulty-case/optimistic/"
        write_file(file_path, output_path, replica_image, with_fault, simulation_name, n_replicas, n_clients, frequency, n_operations, True, False)

        output_path = "results/faulty-case/consistent/"
        file_path = "simulation-files/faulty-case/consistent/"
        write_file(file_path, output_path, replica_image, with_fault, simulation_name, n_replicas, n_clients, frequency, n_operations, False, True)

        output_path = "results/faulty-case/both/"
        file_path = "simulation-files/faulty-case/both/"
        write_file(file_path, output_path, replica_image, with_fault, simulation_name, n_replicas, n_clients, frequency//2, n_operations//2, True, True)

        #########################
        ## Vanilla Faulty-Case ##
        #########################
    
        replica_image = "moirai-replica-vanilla:dev"
        with_fault = True
        simulation_name = f"vanilla-awset-{n_replicas}r-{n_clients}c-{throughput}f"

        output_path = "results/faulty-case/optimistic/"
        file_path = "simulation-files/faulty-case/optimistic/"
        write_file(file_path, output_path, replica_image, with_fault, simulation_name, n_replicas, n_clients, frequency, n_operations, True, False)

        output_path = "results/faulty-case/consistent/"
        file_path = "simulation-files/faulty-case/consistent/"
        write_file(file_path, output_path, replica_image, with_fault, simulation_name, n_replicas, n_clients, frequency, n_operations, False, True)

        output_path = "results/faulty-case/both/"
        file_path = "simulation-files/faulty-case/both/"
        write_file(file_path, output_path, replica_image, with_fault, simulation_name, n_replicas, n_clients, frequency//2, n_operations//2, True, True)